# Setup

In [ ]:
!pip install pyarrow
!pip install plotly
!pip install kaleido

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Image, display

DATA = Path('/home/nik/workspace/ImperialWork/msc_project/SyncNetProject/data')
DATASET_FIGS = Path('./outputs/figures/chap:datasets')
RESULT_FIGS = Path('./outputs/figures/chap:results')
TABLES = Path('./outputs/tables')

for d in (DATASET_FIGS, RESULT_FIGS, TABLES):
    d.mkdir(parents=True, exist_ok=True)

COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2']


def show(fig, name, width=640, height=420):
    fig.update_layout(
        width=width, height=height, template='simple_white', font=dict(family='serif', size=13),
        margin=dict(l=50, r=20, t=30, b=45), legend=dict(orientation='h', yanchor='bottom', y=1.02, x=0)
        )
    
    fig.write_image(RESULT_FIGS / f'{name}.png', scale=3)
    fig.show()


def write_tex(name, text):
    (TABLES / f'{name}.tex').write_text(text)
    print(text)

In [ ]:
MAIN = {
    'sort_of_clevr': {
        'SyncNet': '24a696e62a',
        'Q-Only': '5c938564b4',
        'CNN+MLP': 'ace54f4218',
        'RelNet': 'f59547d9ec',
        'FiLM': '2fe4b0dd67',
        'Transformer': ['f480f82d5a', '852665df79'],
        'Workspace': ['ef208275ef', 'fb142d6dd9'],
    },
    'sqoop': {
        'SyncNet': {1: ['26e9afa0c8', '4937b0b2bd'], 2: '58ec8832e8', 4: 'a3952e47c1', 8: 'a974fb6b3d', 18: 'e375647fd3', 35: 'ee2e99f2fb'},
        'Q-Only': {1: 'da868e47f3', 2: '8c385b2059', 4: 'b53f5ecb08', 8: 'c7cfb0dd1c', 18: 'd640141c57', 35: 'a09acf017b'},
        'Conv+LSTM': {1: ['4dd2f8a5f6', '030e2fbb5c'], 2: ['b838865278', '2bf9832aac'], 4: ['11380cec6a', '1b4f721c75'], 8: ['125ce8d50e', '6b2957d7de'], 18: ['6828367e4c', '5f2e70cb1e'], 35: ['d6cbb5b004', 'a2ff50a58f']},
        'RelNet': {1: ['d91b42c323', 'b542189d54'], 2: ['41515ce582', 'ab085716db'], 4: ['46ef70288f', '44d2c2dc89'], 8: ['f15b7d0159', '2d24259822'], 18: ['fd877702cb', 'e61bdb6b4b'], 35: ['18a92d8762', '62afe001d2']},
        'FiLM': {1: ['48c351e38b', '73bc0a0e0d'], 2: ['cbbfbd61a7', '52c99bc7b5'], 4: ['13e08caaa4', 'c61805702d'], 8: ['70f71488ff', 'd4892fa784'], 18: ['4cf87adb37', '187a5c81b5'], 35: ['dd57cc6026', '5909eef4cc']},
        'Transformer': {1: '0d1162f980', 2: '30f0e4636f', 4: '309d772f74', 8: '12f14a1e08', 18: '556651c974', 35: 'cb057ccfe6'},
        'Workspace': {1: '53921fc0cc', 2: '982e336df1', 4: '5e11bd62b0', 8: '402a405105', 18: '6cc38fc0d1', 35: 'cc51c8cefc'},
    },
}

RELNET_TUNED = {1: '2730963706', 2: 'e80933c8ae', 4: 'd5b9d0fc01', 8: '1531d20844', 18: ['6fa566cfca', 'da8c0421ca', '3964543d7f'], 35: '668b59aaa1'}
CONV_TUNED = {1: '2068622631', 2: '3df6695167', 4: '01622d3cab', 8: '3e4420321b', 18: '664fb0485f', 35: 'a8870d297d'}

ABLATIONS = {
    'sort_of_clevr': {
        'canonical': '24a696e62a',
        'no_bias': '6db60b8be7',
        'shared_cells': '7179abee63',
        'lines_attention': 'beccaa2708',
        'softmax_read': '936b785378',
        'content_addresses': '9b2747a911',
    },
    'sqoop': {
        'canonical': ['26e9afa0c8', '4937b0b2bd'],
        'no_bias': 'c0f9b9b851',
        'shared_cells': '124045c7d5',
        'lines_attention': '316ada3c53',
        'softmax_read': 'c088446626',
        'content_addresses': '0363958211',
    },
}

SWEEPS = {
    'sort_of_clevr': {
        't_train': {2: '356a8b1b86', 4: 'bd355a8480', 8: '24a696e62a', 16: '1acd79d3de'},
        'phase_dim': {2: 'e77dfa1fa0', 3: '9bf1e1d99b', 4: 'c200e19116', 6: '24a696e62a', 8: '1af61137e6'},
        'n_modules': {4: '48436067bf', 6: '24a696e62a', 8: '0d6632bf58', 12: '5e2356e12e'},
    },
    'sqoop': {
        't_train': {2: '1ebcbbdcbb', 4: '0d1776d166', 8: ['26e9afa0c8', '4937b0b2bd'], 16: '6ccdc34b22'},
        'phase_dim': {2: 'ab8b2b8058', 3: 'b313fa2755', 4: '45ffead3af', 6: ['26e9afa0c8', '4937b0b2bd'], 8: '756fda117c'},
        'n_modules': {4: 'c244c32db2', 6: ['26e9afa0c8', '4937b0b2bd'], 8: '537ed77c7b', 12: 'f29ffaf993'},
    },
}

In [ ]:
import wandb

PULL = False

CACHE = Path('.cache/wandb')
CACHE.mkdir(parents=True, exist_ok=True)

RUNS = {}

for task in ('sort_of_clevr', 'sqoop'):

    cache = CACHE / f'{task}_v16.pkl'

    if PULL or not cache.exists():

        rows = [
            {
                'Created': str(r.created_at),
                **r.config,
                **{k: v for k, v in r.summary.items() if isinstance(v, (int, float))}
            }
            for r in wandb.Api(timeout=120).runs(
                f'niks_priv/{task}',
                filters={'state': 'finished'}
            )
        ]

        df = (
            pd.json_normalize(rows)
            .sort_values('Created')
            .drop_duplicates(['cfg_hash', 'train.seed'], keep='last')
        )

        df.to_pickle(cache)

    else:

        df = pd.read_pickle(cache)

    RUNS[task] = df


def runs(task, cfg_hash):

    df = RUNS[task]

    hashes = cfg_hash if isinstance(cfg_hash, list) else [cfg_hash]

    return df[df.cfg_hash.isin([h for h in hashes if h])]


def stat(task, cfg_hash, col):

    r = runs(task, cfg_hash)

    v = 100 * r[col].dropna() if col in r else pd.Series(dtype=float)

    return (v.mean(), v.std(ddof=1) if len(v) > 1 else 0.0, len(v)) if len(v) else (np.nan, np.nan, 0)


def ms(task, cfg_hash, col):

    m, s, k = stat(task, cfg_hash, col)

    return '---' if np.isnan(m) else f'{m:.1f}'


def msd(task, cfg_hash, col):

    m, s, k = stat(task, cfg_hash, col)

    return '---' if np.isnan(m) else f'{m:.1f} $\\pm$ {s:.1f}'

# Chapter 4

In [ ]:
import matplotlib.pyplot as plt

soc = np.load(DATA / 'sort-of-clevr' / 'test.npz', allow_pickle=False)

plt.imsave(DATASET_FIGS / 'soc_example.png', soc['images'][0][..., ::-1])
display(Image(DATASET_FIGS / 'soc_example.png'))

In [ ]:
sq = np.load(DATA / 'sqoop-rhs18-n1080000' / 'val_seen.npz', allow_pickle=True)

for i in range(2):
    plt.imsave(DATASET_FIGS / f'sqoop_example_{i}.png', np.kron(sq['images'][i], np.ones((6, 6, 1), dtype=np.uint8)))
    display(Image(DATASET_FIGS / f'sqoop_example_{i}.png'))

# Chapter 5

In [ ]:
HEADS = {
    'SyncNet': r'\textsc{SyncNet}', 'Q-Only': r'\textit{Q-Only}', 'CNN+MLP': r'\textsc{CNN+MLP}', 'RelNet': r'\textsc{RelNet}',
    'FiLM': r'\textsc{FiLM}', 'Transformer': r'\textsc{Trans.}', 'Workspace': r'\textsc{Works.}'
    }
FAMILIES = {
    'Overall': 'test_callbacks/accuracy', 'Non-relational': 'test_callbacks/non_relational_accuracy',
    'Binary': 'test_callbacks/binary_accuracy', 'Ternary': 'test_callbacks/ternary_accuracy'
    }
SUBTYPES = {
    'Non-relational': {'Query shape': 'query_shape', 'Left of centre': 'left_of_centre', 'Top half': 'top_half'},
    'Binary': {'Closest shape': 'closest_shape', 'Furthest shape': 'furthest_shape', 'Count same shape': 'count_same_shape'},
    'Ternary': {'Count in box': 'count_in_box', 'On band': 'on_band', 'Count obtuse': 'count_obtuse'},
}

models = list(MAIN['sort_of_clevr'])

def soc_row(name, col, bold=False):
    label = rf'\textbf{{{name}}}' if bold else rf'\hspace{{1em}}{name}'
    return label + ' & ' + ' & '.join(ms('sort_of_clevr', MAIN['sort_of_clevr'][m], col) for m in models) + r' \\'


params = ' & '.join(
    f"{runs('sort_of_clevr', MAIN['sort_of_clevr'][m]).n_params.iloc[0] / 1e6:.2f}M"
    if len(runs('sort_of_clevr', MAIN['sort_of_clevr'][m])) else '---'
    for m in models
)

lines = [rf'\textbf{{Params}} & {params} \\', r'\midrule', soc_row('Overall', FAMILIES['Overall'], True)]

for fam, subs in SUBTYPES.items():
    lines += [r'\midrule', soc_row(fam, FAMILIES[fam], True)]
    lines += [soc_row(nm, f'test_callbacks/{k}_accuracy') for nm, k in subs.items()]

write_tex('soc_results', '\n'.join([
    r'\begin{tabular}{@{}l' + 'c' * len(models) + '@{}}', r'\toprule',
    '& ' + ' & '.join(HEADS[m] for m in models) + r' \\', rf'\cmidrule(l){{2-{len(models) + 1}}}',
    *lines, r'\bottomrule', r'\end{tabular}']))

In [ ]:
ROWS = [('canonical', r'\textsc{SyncNet} (canonical)'), ('no_bias', 'no spatial bias'), ('shared_cells', 'cells shared'),
        ('lines_attention', 'private lines, attention gate'), ('softmax_read', 'softmax read'), ('content_addresses', 'content addresses')]


def abl_row(key, label):
    h, hq = ABLATIONS['sort_of_clevr'][key], ABLATIONS['sqoop'][key]
    fams = ' & '.join(ms('sort_of_clevr', h, FAMILIES[f]) for f in ('Non-relational', 'Binary', 'Ternary'))
    return f"{label} & {msd('sort_of_clevr', h, 'test_callbacks/accuracy')} & {fams} & {ms('sqoop', hq, 'test_callbacks/accuracy')}" + r' \\'


write_tex('ablations', '\n'.join([
    r'\begin{tabular}{@{}lccccc@{}}', r'\toprule',
    r'& \multicolumn{4}{c}{\textbf{Sort-of-CLEVR}} & \textbf{SQOOP} \\', r'\cmidrule(lr){2-5}\cmidrule(l){6-6}',
    r'\textbf{Ablation} & \textbf{Overall} & \textbf{Non-rel.} & \textbf{Binary} & \textbf{Ternary} & \textbf{Held-out ($1$)} \\', r'\midrule',
    abl_row(*ROWS[0]), r'\midrule', *[abl_row(*r) for r in ROWS[1:]], r'\bottomrule', r'\end{tabular}']))

In [ ]:
DROPS = {'Freeze': 'test_override/freeze_phase_drop', 'Zero': 'test_override/zero_phase_drop', 'Shuffle': 'test_override/shuffle_phase_drop'}
TASKS = {'Sort-of-CLEVR': ('sort_of_clevr', MAIN['sort_of_clevr']['SyncNet']), 'SQOOP ($1$)': ('sqoop', MAIN['sqoop']['SyncNet'][1])}

body = [
    f"{label} & " + ' & '.join(('---' if msd(task, h, col) == '---' else '-' + msd(task, h, col)) for col in DROPS.values()) + r' \\'
    for label, (task, h) in TASKS.items()
]

write_tex('overrides', '\n'.join([
    r'\begin{tabular}{@{}lccc@{}}', r'\toprule',
    r'\textbf{Task} & ' + ' & '.join(rf'\textbf{{{k}}}' for k in DROPS) + r' \\', r'\midrule',
    *body, r'\bottomrule', r'\end{tabular}']))

In [ ]:
LABELS = {'Q-Only': r'\textit{Q-Only}', 'Conv+LSTM': r'\textsc{Conv+LSTM}', 'RelNet': r'\textsc{RelNet}', 'FiLM': r'\textsc{FiLM}',
          'Transformer': r'\textsc{Transformer}$^\dagger$', 'Workspace': r'\textsc{Workspace}$^\dagger$', 'SyncNet': r'\textsc{SyncNet}'}
SPLITS = {'train': 'train_callbacks/accuracy', 'validation': 'eval_callbacks/accuracy', 'held-out': 'test_callbacks/accuracy'}


def sq_row(label, cells):
    lines = [' & '.join(msd('sqoop', cells.get(v), col) for v in (1, 2, 4, 8, 18, 35)) for col in SPLITS.values()]
    first = rf'\multirow{{3}}{{*}}{{{label}}} & train & {lines[0]} \\'
    rest = [rf' & {split} & {line} \\' for split, line in zip(list(SPLITS)[1:], lines[1:])]
    return '\n'.join([first, *rest])


rows = [sq_row(LABELS[m], MAIN['sqoop'][m]) for m in ('Q-Only', 'Conv+LSTM', 'RelNet', 'FiLM', 'Transformer', 'Workspace', 'SyncNet')]
rows += [sq_row(r'\textsc{RelNet}$^\ddagger$', RELNET_TUNED), sq_row(r'\textsc{Conv+LSTM}$^\ddagger$', CONV_TUNED)]

write_tex('sqoop_results', '\n'.join([
    r'\begin{tabular}{@{}llcccccc@{}}', r'\toprule',
    r'& & \multicolumn{6}{c}{\textbf{Pair variety}} \\', r'\cmidrule(l){3-8}',
    r'\textbf{Model} & \textbf{Split} & ' + ' & '.join(f'${v}$' for v in (1, 2, 4, 8, 18, 35)) + r' \\', r'\midrule',
    rows[0], r'\midrule', *('\n' + r'\addlinespace' + '\n').join(rows[1:]).split('\n'),
    r'\bottomrule', r'\end{tabular}']))

In [ ]:
FIG_MODELS = {
    'Q-Only': MAIN['sqoop']['Q-Only'],
    'Conv+LSTM\u2021': CONV_TUNED,
    'RelNet\u2021': RELNET_TUNED,
    'FiLM': MAIN['sqoop']['FiLM'],
    'Transformer': MAIN['sqoop']['Transformer'],
    'Workspace': MAIN['sqoop']['Workspace'],
    'SyncNet': MAIN['sqoop']['SyncNet'],
}

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.06,
                    subplot_titles=('validation (seen pairs)', 'held-out (unseen pairs)'))

for r, col in enumerate(('eval_callbacks/accuracy', 'test_callbacks/accuracy'), start=1):
    for i, m in enumerate(FIG_MODELS):
        mean, sd, n = zip(*[stat('sqoop', FIG_MODELS[m].get(v), col) for v in (1, 2, 4, 8, 18, 35)])
        fig.add_trace(go.Bar(x=['1', '2', '4', '8', '18', '35'], y=mean, name=m, marker_color=COLORS[i % len(COLORS)],
                             legendgroup=m, showlegend=(r == 1), error_y=dict(type='data', array=sd, visible=True, thickness=1),
                             customdata=np.c_[sd, n],
                             hovertemplate=f'{m}<br>pair variety %{{x}}<br>%{{y:.1f}} ± %{{customdata[0]:.1f}} (n=%{{customdata[1]}})<extra></extra>'),
                      row=r, col=1)
    fig.add_hline(y=50, line=dict(color='grey', dash='dot', width=1), row=r, col=1)
    fig.update_yaxes(title='Accuracy (%)', range=[40, 101], row=r, col=1)

fig.update_xaxes(title='Pair variety', row=2, col=1)
fig.update_layout(barmode='group', bargap=0.15,
                  legend=dict(orientation='h', yanchor='bottom', y=1.06, x=0, font=dict(size=11), itemwidth=30),
                  margin=dict(l=50, r=20, t=70, b=45))

show(fig, 'sqoop_results_bars_stacked', width=820, height=560)

In [ ]:
def sweep(axis, xlabel, canonical, name):
    fig = go.Figure()

    series = [(fam, 'sort_of_clevr', SWEEPS['sort_of_clevr'][axis], col, dict(color=COLORS[i]))
              for i, (fam, col) in enumerate(FAMILIES.items())]
    series += [('SQOOP (1)', 'sqoop', SWEEPS['sqoop'][axis], 'test_callbacks/accuracy', dict(color='black', dash='dash'))]

    for label, task, cells, col, line in series:
        pts = [(x, *stat(task, h, col)[:2]) for x, h in cells.items()]
        pts = [p for p in pts if not np.isnan(p[1])]

        if not pts:
            continue

        x, m, s = map(np.array, zip(*pts))
        fig.add_trace(go.Scatter(x=np.r_[x, x[::-1]], y=np.r_[m + s, (m - s)[::-1]], fill='toself', fillcolor=line['color'],
                                 opacity=0.12, line=dict(width=0), showlegend=False, hoverinfo='skip'))
        fig.add_trace(go.Scatter(x=x, y=m, mode='lines+markers', name=label, line=line,
                                 marker=dict(size=6, symbol='square' if task == 'sqoop' else 'circle'), customdata=s,
                                 hovertemplate=f'{label}<br>{xlabel}=%{{x}}<br>%{{y:.1f}} ± %{{customdata:.1f}}<extra></extra>'))

    fig.add_vline(x=canonical, line=dict(color='grey', dash='dot', width=1))
    fig.update_xaxes(title=xlabel, tickvals=list(SWEEPS['sort_of_clevr'][axis]))
    fig.update_yaxes(title='Accuracy (%)')
    show(fig, name, width=520, height=380)


sweep('t_train', 'Internal steps T', 8, 'soc_ablation_t_train')
sweep('n_modules', 'Number of modules M', 6, 'soc_ablation_n_modules')
sweep('phase_dim', 'Phase dimension d', 6, 'soc_ablation_phase_dim')